<a href="https://colab.research.google.com/github/Ganasa18/belajar-tensorflow/blob/main/train_class_prompt_labeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets pandas requests tqdm scikit-learn sentence-transformers joblib

In [ ]:
from datasets import load_dataset

ds = load_dataset(
    "ilhamfadheel/alpaca-cleaned-indonesian",
    split="train"
)

print(ds)
print(ds[0])

In [ ]:
# IMPORT DATA SET
SAMPLE_SIZE = 5000

ds_small = ds.shuffle(seed=42).select(
    range(min(SAMPLE_SIZE, len(ds)))
)

def build_prompt(row):
    instruction = row["instruction"].strip()
    input_text = row["input"].strip()

    if input_text:
        return instruction + "\n" + input_text

    return instruction

prompts = [build_prompt(x) for x in ds_small]

print("Jumlah prompt:", len(prompts))

for x in prompts[:5]:
    print("\n---")
    print(x)

In [ ]:
# SIMPLE
# Fakta sederhana, lookup, klasifikasi sederhana,
# jawaban singkat, operasi mudah.

# GENERAL
# Penjelasan umum yang tidak membutuhkan reasoning berat.

# REASONING
# Analisis, matematika, planning, comparison,
# trade-off, deduksi, multi-step problem.

# CODING_SIMPLE
# Generate snippet sederhana, regex, SQL sederhana,
# fungsi kecil, syntax.

# CODING_COMPLEX
# Debugging kompleks, architecture, multi-file,
# race condition, performance, security review.

# TRANSFORM
# Translation, summarization, rewrite,
# extract, shorten, formatting.

# CREATIVE
# Story, brainstorming kreatif, slogan,
# dialog, ide kreatif.

LABELS = [
    "SIMPLE",
    "GENERAL",
    "REASONING",
    "CODING_SIMPLE",
    "CODING_COMPLEX",
    "TRANSFORM",
    "CREATIVE"
]

In [ ]:
import os

BASE_URL = "https://YOUR-ENDPOINT/v1/chat/completions"
API_KEY = "ISI_API_KEY"
MODEL = "MODEL_TEACHER_KAMU"

TEMPERATURE = 0
TIMEOUT = 90

In [ ]:
from getpass import getpass

API_KEY = getpass("API Key: ")

In [ ]:
SYSTEM_PROMPT = """
You are labeling user prompts for an AI model router.

Classify each prompt into EXACTLY ONE label:

SIMPLE
- Simple factual questions
- Easy lookup
- Basic classification
- Very short/simple task
- Does not require substantial reasoning

GENERAL
- General explanation
- Normal knowledge question
- Moderate general assistant task
- Does not clearly belong to another category

REASONING
- Mathematics
- Multi-step reasoning
- Planning
- Comparison and trade-offs
- Analysis
- Deduction
- Complex decision making

CODING_SIMPLE
- Small code snippets
- Simple functions
- Regex
- Basic SQL
- Syntax questions
- Straightforward programming tasks

CODING_COMPLEX
- Complex debugging
- Software architecture
- Multi-component systems
- Race conditions
- Security/code review
- Performance optimization
- Complex implementation

TRANSFORM
- Translation
- Summarization
- Rewrite
- Shortening
- Extraction
- Reformatting
- Changing an existing text without requiring substantial new reasoning

CREATIVE
- Story writing
- Creative brainstorming
- Dialogue
- Slogans
- Fiction
- Creative ideation

Important rules:

1. Classify based on USER INTENT, not individual keywords.
2. The prompt may be Indonesian, English, or mixed Indonesian-English.
3. Technical words do NOT automatically mean CODING.
4. Mentioning a language such as "Bahasa Inggris" does NOT automatically mean TRANSFORM.
5. "Explain" does NOT automatically mean REASONING.
6. Choose CODING_COMPLEX only if strong technical reasoning or debugging is actually required.
7. Use SIMPLE for genuinely easy tasks.
8. Return valid JSON only.

Output schema:

{
  "label": "ONE_LABEL",
  "difficulty": 1,
  "confidence": 0.95
}

difficulty must be integer 1-5.
confidence must be number 0-1.
"""

In [ ]:
# Call API

import requests
import json
import time

def call_teacher(prompt):
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": MODEL,
        "temperature": TEMPERATURE,
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    }

    response = requests.post(
        BASE_URL,
        headers=headers,
        json=payload,
        timeout=TIMEOUT
    )

    response.raise_for_status()

    data = response.json()

    return data["choices"][0]["message"]["content"]

In [ ]:
# PARSER JSON

import re

def parse_teacher_output(text):
    text = text.strip()

    text = re.sub(
        r"^```(?:json)?\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\s*```$",
        "",
        text
    )

    match = re.search(r"\{.*\}", text, re.DOTALL)

    if not match:
        raise ValueError(
            f"Tidak menemukan JSON: {text}"
        )

    data = json.loads(match.group())

    label = data.get("label")
    difficulty = data.get("difficulty")
    confidence = data.get("confidence")

    if label not in LABELS:
        raise ValueError(
            f"Label invalid: {label}"
        )

    difficulty = int(difficulty)
    confidence = float(confidence)

    if difficulty < 1 or difficulty > 5:
        raise ValueError("difficulty di luar range")

    if confidence < 0 or confidence > 1:
        raise ValueError("confidence di luar range")

    return {
        "label": label,
        "difficulty": difficulty,
        "confidence": confidence
    }

In [ ]:
test_prompt = """
review architecture backend saya dan cari kemungkinan
race condition serta bottleneck performance
"""

raw = call_teacher(test_prompt)

print("RAW:")
print(raw)

parsed = parse_teacher_output(raw)

print("\nPARSED:")
print(parsed)